# Test 3 — Trigger por ADC + barrido de dead-time

Mide la **resolución par-pulso** del sistema multitrigger: a qué
distancia mínima entre pulsos consecutivos el FPGA + SW empiezan a
perder eventos.

Estrategia:
1. DAC genera una **onda cuadrada** continua en OUT1; cada flanco
   ascendente es un "pulso" que queremos detectar.
2. Configuro el scope con OR_MASK = ADC ch0 posedge (`bit 1`) +
   `adc_we_keep=1` (modo continuo).
3. Capturo N eventos consecutivos poleando `wp_trig` (cambia cada
   trigger nuevo) y pulsando `trig_dis_clr` para re-habilitar.
4. Comparo eventos detectados vs esperados (= freq × tiempo).
5. Barro freqs de 1 kHz a 1 MHz: la freq donde la eficiencia cae
   bajo 100 % marca el dead-time.

In [ ]:
import time
import numpy as np
from matplotlib import pyplot as plt
import rp

# Toda la lógica de scope/captura/diagnóstico vive en multitrigger_utils.py
from multitrigger_utils import (
    MultiTriggerScope, decode_snap,
    OR_MASK_ALL,
    BIT_SW, BIT_ADC_P0, BIT_ADC_N0, BIT_ADC_P1, BIT_ADC_N1,
    BIT_EXT_P, BIT_EXT_N, BIT_ASG_P, BIT_ASG_N,
    N_BUF, FS,
    events_to_intervals, efficiency, pulses_from_buffer,
)

rp.rp_Init()
sc = MultiTriggerScope.open()
print('Scope abierto. cfg sanity:',
      f'trg_src ch0 @0x240 = {sc.r32(0x240):#010x}',
      f'(esperado tras reset: 0)')

In [ ]:
# Carga del bitstream (sin device tree — vía fpgautil)
!fpgautil -b /root/red_pitaya_top.bit.bin
print(f'cfg sanity: trg_src ch0 @0x240 = {sc.r32(0x240):#010x}  (esperado tras reset: 0)')

## Generador externo: Rigol DG4162

La señal de pulsos viene de un **Rigol DG4162** controlado por SCPI vía
USB-TMC (driver kernel `/dev/usbtmcN`) o TCP. El driver está en
[`rigol_dg4162.py`](./rigol_dg4162.py).

Dos modos:

- **Pulsos periódicos** (`set_pulse_periodic(period_s, width_s, ...)`):
  un pulso cada `period_s`. Cada flanco ascendente es un trigger.
  Cambiar el período = cambiar la distancia entre pulsos.
- **Pares de pulsos en burst** (`set_pulse_pair_burst(gap_s, burst_period_s, ...)`):
  dos pulsos separados por `gap_s`, repetidos cada `burst_period_s`.
  Mide directamente la **pulse-pair resolution**.

Cableado: Rigol OUT1 → Pitaya IN1. Jumper LV (1:1).

In [ ]:
# Importar driver del Rigol y abrir conexión.
# Si tu Rigol está en otro device (p.ej. /dev/usbtmc1) o por TCP, cambiá esto.
from rigol_dg4162 import RigolDG4162

#try:
#    rg = RigolDG4162.usbtmc('/dev/usbtmc0')
#except FileNotFoundError:
    # Fallback TCP — poné la IP del Rigol
rg = RigolDG4162.vxi11('10.73.28.35')
print('Conectado:', rg.id)

# Configuración base: pulsos angostos (200 ns) de 0 a 1 V.
# Período inicial 100 µs (= 10 kHz). Se cambia con rg.set_pulse_period().
RIGOL_CH        = 1
PULSE_WIDTH_S   = 200e-9
PULSE_AMP_VPP   = 1.0
PULSE_OFFSET_V  = 0.5      # nivel base = 0 V, peak = 1 V

rg.reset()
rg.set_pulse_periodic(ch=RIGOL_CH, period_s=100e-6,
                       width_s=PULSE_WIDTH_S,
                       amp_vpp=PULSE_AMP_VPP, offset_v=PULSE_OFFSET_V)
rg.output(RIGOL_CH, True)
print('Rigol armado: pulsos a 10 kHz, ancho 200 ns, 0–1 V')
print('  último error SCPI:', rg.check_error())

## Diagnóstico

Si una captura da `capturados: 0` (o cualquier resultado raro), corré esta
celda para ver dónde se corta la cadena. Hace tres pasos:

1. `sc.debug_trigger()` en IDLE → estado base de todos los regs.
2. `sc.acq_capture_sw(thr)` + plot del buffer IN1 → confirma que la señal
   del Rigol llega a la entrada y el `bram_sm` captura un buffer
   consistente (independiente del trigger ADC).
3. `sc.arm_for_adc_trigger(...)` + sleep 100 ms + `sc.debug_trigger()` →
   muestra si el FPGA disparó (snapshot ≠ 0), si la máscara latcheó, si
   we_keep quedó en `0x3` y si `wp_trig` avanzó.

Qué buscar:
- **`snapshot == 0`** tras el arm + sleep → ningún flanco cruzó el threshold.
  Causa: threshold mal, mask mal, señal por debajo del umbral.
- **`dis_act != 0` y quieto en 1** → adc_trg_dis se trabó (problema del
  trig_dis_clr / shield).
- **`we_keep = 0x0`** después del arm → el fix del orden no entró
  (re-cargá `multitrigger_utils.py`).
- **`wp_cur` avanza pero `wp_trig` quieto** → FSM escribe el buffer pero
  el trigger no firma.

In [ ]:
print('=== 1. Estado en IDLE ===')
sc.debug_trigger()

print('\n=== 2. Captura forzada por SW trigger (sanity de buffer + señal) ===')
d1, d2, snap = sc.acq_capture_sw(thr=0.5)
print(f'  snap = {snap:#010x} -> {decode_snap(snap)}   (esperado: sw_any)')
print(f'  IN1 pp = {d1.max()-d1.min():.3f} V   media = {d1.mean():+.3f} V')
print(f'  IN1 max = {d1.max():.3f} V   min = {d1.min():.3f} V   '
      f'(esperado pp ~1 V, mean ~0.5 V con offset Rigol)')

plt.figure(figsize=(10, 3))
plt.plot(d1[:2000], label='IN1 (Rigol)')
plt.axhline(0.5, color='r', ls='--', lw=0.8, label='threshold 0.5 V')
plt.xlabel('sample'); plt.ylabel('V')
plt.title(f'SW trigger sanity — pp={d1.max()-d1.min():.2f} V')
plt.legend(); plt.grid(True); plt.show()

print('\n=== 3. Arm ADC trigger ch0 posedge, esperar 100 ms ===')
sc.arm_for_adc_trigger(mask_ch0=BIT_ADC_P0, mask_ch1=BIT_ADC_P0, thr=0.5)
print('  estado JUSTO después del arm:')
sc.debug_dump()
time.sleep(0.1)
print('\n  estado DESPUÉS de 100 ms:')
sc.debug_trigger()
sc.disarm()

## Captura de N eventos consecutivos

Las funciones viven en [`multitrigger_utils.py`](./multitrigger_utils.py) y se
usan vía la instancia `sc` (`MultiTriggerScope`):

- `sc.arm_for_adc_trigger(mask_ch0, mask_ch1, thr, hyst, delay, we_keep_both, auto_rearm)`:
  configura decim/threshold/delay/hyst + OR_MASK + arm + we_keep.

  **`auto_rearm=True`** (default): activa el `trigger_shield` para clear
  automático del `adc_trg_dis` (shield_dur=0, mismo ciclo) Y sobreescribe
  `set_dly=0` raw (saltea el offset de ~65 µs que mete
  `rp_AcqSetTriggerDelay`). Con esto el dead-time del stack baja a la
  región de pocos µs.

  **`auto_rearm=False`**: comportamiento legacy. El SW tiene que pulsar
  `0x94` entre triggers y el `bram_sm` tiene el offset de 65 µs.

  **OJO**: la API `rp_AcqStart` borra el `we_keep` como side-effect (escribe
  byte0=0x01 en 0x00, lo que en la cfg satisface `|sys_dats` y resetea el
  bit 3). Por eso el método setea we_keep DESPUÉS del arm.

- `sc.capture_n_events(n, timeout_ms, wp_addr, clear_both)`: polea cambios
  de `wp_trig @0x1C`. Con `auto_rearm=True` el shield limpia `adc_trg_dis`
  por HW, así que el loop solo lee `wp_trig` (mínimo overhead).

- `sc.disarm()`: apaga we_keep, desactiva el shield, limpia
  `adc_trg_dis` y resetea el FSM.

- `events_to_intervals(events)`: intervalos SW (µs) entre triggers.
- `efficiency(observed_n, target_freq, duration)`: ratio capturados / esperados.

## Ejemplo: una captura de N eventos

Rigol genera pulsos cada 100 µs en OUT1 (= 10 kHz, ancho 200 ns).
Threshold del scope a 0.5 V (mitad de la altura del pulso).
Esperamos un trigger por cada pulso → dt esperado = 100 µs.

**Si `capturados = 0`**, correr la celda **Diagnóstico** de arriba para ver
dónde se rompe (señal, mask, threshold, we_keep).

In [ ]:
PERIOD_S = 100e-6   # 10 kHz → 100 µs entre pulsos
rg.set_pulse_period(ch=RIGOL_CH, period_s=PERIOD_S)
# auto_rearm=True (default) activa el trigger_shield para clear automático
# del adc_trg_dis + sobreescribe set_dly=0 raw (saltea el offset de 65 µs
# que mete rp_AcqSetTriggerDelay). Sin esto el dead-time floor es ~65 µs.
sc.arm_for_adc_trigger(mask_ch0=BIT_ADC_P0, mask_ch1=BIT_ADC_P0,
                        thr=0.5, auto_rearm=True)

events, dur_s = sc.capture_n_events(n=50, timeout_ms=1000)
sc.disarm()

intervals = events_to_intervals(events)
expected_dt = PERIOD_S * 1e6
print(f'capturados: {len(events)} eventos en {dur_s*1e3:.1f} ms')
if len(intervals):
    print(f'dt SW (µs): media={intervals.mean():.2f}  std={intervals.std():.2f}  '
          f'min={intervals.min():.2f}  max={intervals.max():.2f}')
    print(f'esperado    : {expected_dt:.2f} µs')
    print(f'eficiencia  : {efficiency(len(events), 1.0/PERIOD_S, dur_s)*100:.1f}%')
snaps = set(e['snap'] for e in events)
print(f'snapshots únicos: {snaps} -> {[decode_snap(s) for s in snaps]}')

if len(intervals):
    plt.figure(figsize=(10, 3))
    plt.plot(intervals, 'o-')
    plt.axhline(expected_dt, color='r', ls='--', label=f'esperado {expected_dt:.1f} µs')
    plt.xlabel('evento #'); plt.ylabel('dt entre triggers (µs)')
    plt.legend(); plt.grid(True); plt.title(f'Captura N=50, Rigol @{PERIOD_S*1e6:.0f} µs')
    plt.show()

## Barrido de distancia entre pulsos

Para cada distancia esperada (`period_s`):
1. Reconfigurar el Rigol con `set_pulse_period`.
2. Armar el scope (idem cada iteración: depende del threshold/decim).
3. Capturar N eventos (o esperar timeout).
4. Medir eficiencia = `len(eventos) / (1/period_s × duración)`.

La distancia más chica donde la eficiencia todavía es ~100% es la
**resolución par-pulso del stack completo (FPGA + bus + SW)**.

In [ ]:
def sweep_periods(periods_s, n_events=200, timeout_ms=2000,
                   thr=0.5, settle_s=0.1, auto_rearm=True, verbose=True):
    """Barrido en periods_s (lista de períodos en segundos). Por cada uno
    reconfigura el Rigol, arma el scope y captura n_events.

    auto_rearm=True: usa el trigger_shield (clear automático del dis) +
        set_dly=0 raw. Baja el dead-time floor del FPGA de ~65 µs a ~µs.
        Es el modo correcto para medir dead-time real.
    auto_rearm=False: comportamiento legacy (SW pulsa 0x94 cada iteración,
        set_dly mantiene el offset de 65 µs de la rp API). Sirve para
        verificar contra el setup anterior.

    Devuelve list-of-dict con freq, eficiencia, dt medido vs esperado."""
    results = []
    for p in periods_s:
        f = 1.0 / p
        rg.set_pulse_period(ch=RIGOL_CH, period_s=p)
        time.sleep(settle_s)
        sc.arm_for_adc_trigger(mask_ch0=BIT_ADC_P0, mask_ch1=BIT_ADC_P0,
                                thr=thr, auto_rearm=auto_rearm)
        events, dur_s = sc.capture_n_events(n=n_events, timeout_ms=timeout_ms)
        sc.disarm()

        eff = efficiency(len(events), f, dur_s)
        intervals = events_to_intervals(events)
        expected_dt = p * 1e6
        r = {
            'period_s':       p,
            'freq':           f,
            'observed_n':     len(events),
            'duration_s':     dur_s,
            'efficiency':     eff,
            'mean_dt_us':     intervals.mean() if len(intervals) else float('nan'),
            'std_dt_us':      intervals.std()  if len(intervals) else float('nan'),
            'expected_dt_us': expected_dt,
        }
        results.append(r)
        if verbose:
            print(f'p={p*1e6:>9.2f} µs  (f={f:>9.0f} Hz)  '
                  f'n={len(events):>4d}/{n_events}  eff={eff*100:>6.1f}%  '
                  f'dt_med={r["mean_dt_us"]:>8.2f}±{r["std_dt_us"]:>6.2f} µs')
    return results


def plot_deadtime_curve(results):
    fs       = np.array([r['freq'] for r in results])
    effs     = np.array([r['efficiency'] for r in results])
    exp_dt   = np.array([r['expected_dt_us'] for r in results])
    meas_dt  = np.array([r['mean_dt_us']  for r in results])

    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].semilogx(exp_dt, effs * 100, 'o-')
    ax[0].axhline(100, color='gray', ls=':')
    ax[0].axhline(95,  color='orange', ls='--', label='95 %')
    ax[0].set_xlabel('distancia esperada entre pulsos (µs)')
    ax[0].set_ylabel('eficiencia (%)')
    ax[0].set_title('Eficiencia vs distancia entre pulsos')
    ax[0].invert_xaxis()
    ax[0].grid(True, which='both'); ax[0].legend()

    ax[1].loglog(exp_dt, meas_dt, 'o-', label='medido')
    ax[1].plot(exp_dt, exp_dt, 'k--', alpha=0.5, label='ideal y=x')
    ax[1].set_xlabel('dt esperado (µs)')
    ax[1].set_ylabel('dt medido entre triggers (µs)')
    ax[1].set_title('Tiempo medido vs esperado')
    ax[1].grid(True, which='both'); ax[1].legend()
    plt.tight_layout(); plt.show()

    ok = effs > 0.95
    if ok.any():
        f_max_util = fs[ok].max()
        dt_min_util = 1e6 / f_max_util
        print(f'\nFreq máxima con eficiencia >95%: {f_max_util:.0f} Hz')
        print(f'⇒ distancia mínima útil entre pulsos: {dt_min_util:.2f} µs')
    else:
        print('\nNinguna distancia superó 95% de eficiencia.')

In [ ]:
# Barrido de distancias entre pulsos: de 1 ms (fácil) a 1 µs (extremo).
# Cada elemento es el período entre pulsos en segundos.
PERIODS = [1e-3, 500e-6, 200e-6, 100e-6, 50e-6, 20e-6,
           10e-6, 5e-6, 2e-6, 1e-6]

results = sweep_periods(PERIODS, n_events=300, timeout_ms=2000, thr=0.5)
plot_deadtime_curve(results)

## Cómo interpretar la curva

- **Eficiencia (%)** vs distancia esperada:
  - Plateau cerca de 100 % a distancias largas → el sistema captura todo.
  - Caída brusca a distancias chicas → entró en zona de dead-time.
  - La curva idealmente es un escalón (Heaviside) en el dead-time del FPGA.
    En la realidad es suave por jitter SW (Python + bus PS↔PL).
- **dt medido vs esperado** (log-log):
  - Si la diagonal `y=x` matchea → cada flanco del DAC produce 1 trigger.
  - Si la curva se aplana en un valor mínimo → ese valor es el **dead-time SW**
    (no podés ir más rápido que eso desde Python).
  - Si la curva es 2× la diagonal → se pierde 1 de cada 2 flancos.

**Limitación SW**: el polling de `wp_trig` desde Python tarda ~µs por
iteración (típicamente 1–10 µs por loop). Eventos a < 10 µs entre sí van a
perderse aunque el FPGA los vea. Para medir el dead-time del HARDWARE (sin
Python en el loop), habría que hacer un test C/SCPI o usar un contador
interno del FPGA (no incluido en esta versión).

In [ ]:
try:
    sc.disarm()
except Exception as e:
    print('warn disarm:', e)
try:
    rg.output(RIGOL_CH, False)
    rg.close()
except Exception as e:
    print('warn rigol:', e)
sc.close()
rp.rp_Release()
print('cerrado')